# Single-realization dynamics (scratch / exploratory)

Throwaway exploration of the dipole-conserving 0110<->1001 active-site dynamics for a single realization. Not cited in RESEARCH_LOG.md.

Confirmed design choices (2026-09-12):
- A flipped active site remains eligible to flip back immediately if reselected at random on the next step (not consumed by the move).
- `affected` range in the update loop intentionally checks a few extra, unaffected sites beyond the minimal i-3..i+3 window (harmless no-ops).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def findActiveSites():
    active = []

    for i in range(L):
        a = lattice[i]
        b = lattice[(i+1)%L]
        c = lattice[(i+2)%L]
        d = lattice[(i+3)%L]

        if (a,b,c,d)==(1,-1,-1,1):
            active.append(i)

        elif (a,b,c,d)==(-1,1,1,-1):
            active.append(i)
    return active


def is_active(i):
    a = lattice[i]
    b = lattice[(i+1)%L]
    c = lattice[(i+2)%L]
    d = lattice[(i+3)%L]

    return ((a,b,c,d)==(1,-1,-1,1)
         or (a,b,c,d)==(-1,1,1,-1))

def remove(j):
    idx=where[j]
    last=active_sites[-1]

    active_sites[idx]=last
    where[last]=idx

    active_sites.pop()
    where[j]=-1

def add(j):
    active_sites.append(int(j))
    where[j]=len(active_sites)-1

def flip_pattern(i):
    sites = [(i+k) % L for k in range(4)]

    a = lattice[sites[0]]
    b = lattice[sites[1]]
    c = lattice[sites[2]]
    d = lattice[sites[3]]

    if (a,b,c,d) == (1,-1,-1,1):
        new = (-1,1,1,-1)

    elif (a,b,c,d) == (-1,1,1,-1):
        new = (1,-1,-1,1)

    else:
        return False

    for s, val in zip(sites, new):
        lattice[s] = val

    return True

In [ ]:
L = 100
Steps = 100

lattice = np.random.choice([-1,1], size=L, p=[.5, .5])
active_sites = findActiveSites()
where = np.full(L, -1)

history=[lattice.copy()]
active_history=[active_sites.copy()]
active_vs_time = [len(active_sites)]

c=0
for i in active_sites:
    where[i]=c
    c=c+1
for t in range(Steps):
    k = np.random.randint(len(active_sites))
    i = active_sites[k]

    affected = np.arange(i-3, i+7)%L
    flip_pattern(i)

    for j in affected:
        if is_active(j):
            if where[j]==-1:
                add(j)
        else:
            if where[j]!=-1:
                remove(j)
    history.append(lattice.copy())
    active_history.append(active_sites.copy())
    active_vs_time.append(len(active_sites))

In [ ]:
history = np.array(history)
history_01 = (history + 1)//2    # optional: map -1/+1 -> 0/1

fig, ax = plt.subplots(figsize=(8,8))

ax.imshow(history_01,
          cmap='viridis',
          interpolation='none',
          origin='lower',
          aspect='equal')

# Draw grid at every cell boundary
ax.set_xticks(np.arange(-0.5, history.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, history.shape[0], 1), minor=True)
ax.grid(which='minor', color='gray', linewidth=0.5)

# Hide minor tick marks
ax.tick_params(which='minor', bottom=False, left=False)

ax.set_xlabel("Lattice site")
ax.set_ylabel("Time step")

plt.show()